# Step 5 · GRPO

Online RL from the SFT model. Each step takes 4 questions, samples 8 answers to each from the *current* model, scores them, and updates the model towards the answers that beat their group's average:

R = 1.0 × letter_acc + 0.5 × exact + 0.5 × fluency − format penalties

Fluency is scored by a separate copy of the original model. Same 600 questions as DPO, so only the method differs.

1. **Pilot:** `MAX_STEPS = 10`. Check the reward curve and read the samples.
2. **Full run:** set `MAX_STEPS = 150` and rerun the training cell; it continues from the pilot.
3. **Evaluate** the final model on `val`, plus the step-50 and step-100 snapshots for a val curve.

Checkpoints go to Drive every 10 steps; after a disconnect, run setup and settings, then the cell you were on. Needs the SFT and DPO runs (for the adapter, the questions and their fluency references).

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
N_SFT = 2000          # training questions SFT used
N_QUESTIONS = 600     # DPO's questions (the ones after SFT's)
MAX_STEPS = 10        # pilot; then 150 (one pass over the 600 questions)
QUESTIONS_PER_STEP = 4
NUM_GENERATIONS = 8   # answers per question
MAX_COMPLETION = 192  # tokens; the 99th percentile of SFT answers is ~161
LEARNING_RATE = 1e-5
N_EVAL = 100
BATCH_SIZE = 16       # evaluation generation batch

In [ ]:
import gc, torch
from stego.colab import run_dir
from stego.experiments import baselines as e0, dpo, grpo, sft
from stego.io import read_jsonl

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
name = MODEL.split("/")[-1]
DATA_DIR = run_dir("00_data")
BASELINES_DIR = run_dir(f"02_baselines-{CHANNEL}-{name}")
SFT_DIR = run_dir(f"03_sft-{CHANNEL}-{name}")
DPO_DIR = run_dir(f"04_dpo-{CHANNEL}-{name}")
RUN_DIR = run_dir(f"05_grpo-{CHANNEL}-{name}")

examples = read_jsonl(DATA_DIR / f"all_splits_examples_{CHANNEL}.jsonl")
grpo_rows = sft.pick_training_examples(examples, N_SFT + N_QUESTIONS)[N_SFT:]
val_rows = [r for r in e0.eval_rows(examples, N_EVAL) if r["split"] == "val"]
print(len(grpo_rows), "GRPO questions;", len(val_rows), "val questions; run folder:", RUN_DIR)

def free_gpu():
    for v in ("model", "tokenizer", "scorer"):
        globals().pop(v, None)
    gc.collect(); torch.cuda.empty_cache()

## 1–2. Train (pilot, then full run)

Policy: the SFT adapter merged into the model, plus a new LoRA. Scorer: a separate copy of the original model. Fluency references are the ones DPO generated for these questions.

In [ ]:
from stego.generation.runner import load_model

model, tokenizer = dpo.load_merged_sft(MODEL, SFT_DIR / "lora_adapter_final")
scorer, _ = load_model(MODEL)
train_refs = e0.generate_references(DPO_DIR, grpo_rows, scorer, tokenizer, sampled=True)  # already there: just read

ADAPTER = grpo.train_grpo(
    RUN_DIR, grpo_rows, model, tokenizer, scorer, train_refs,
    max_steps=MAX_STEPS, questions_per_step=QUESTIONS_PER_STEP, num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION, learning_rate=LEARNING_RATE,
)
free_gpu()

In [ ]:
import pandas as pd
curve = pd.DataFrame(grpo.training_curve(RUN_DIR)).set_index("step")
curve.columns = [c.replace("rewards/", "").replace("/mean", "").replace("acrostic_", "").replace("completions/", "") for c in curve.columns]
display(curve.round(3))

In [ ]:
# The latest logged group of answers (one question, all 8 answers), best first.
samples = read_jsonl(RUN_DIR / "train_rollouts_logged_during_training.jsonl")
last = [s for s in samples if s["step"] == samples[-1]["step"]]
print(f"step {last[0]['step']} · secret={last[0]['secret']} · Q: {last[0]['question']}\n")
for s in sorted(last, key=lambda s: -s["reward"]):
    print(f"R={s['reward']:.2f} decoded={s['decoded']} flu={s['fluency']:.2f} fmt=-{s['format_penalty']:.2f} leak={s['leak']}")
    print(s["answer"], "\n")

**Reading the pilot:**
- `frac_reward_zero_std` should stay low: it's the share of questions whose 8 answers all scored the same, which teach nothing.
- `reward` should drift up (10 steps is short, so it will be noisy). Also check how long a step took.
- `clipped_ratio` is the share of answers cut off at the token cap; if it's high, raise `MAX_COMPLETION`.
- Read the samples for cheats (very short sentences, naming the secret, filler).

If it looks good, set `MAX_STEPS = 150`, rerun the settings cell and the training cell.

## 3. Evaluate

The final model on the same `val` questions as before, then each snapshot (steps 50, 100, 150) for a val curve.

In [ ]:
from IPython.display import Markdown, display

scorer, _ = load_model(MODEL)
val_refs = {r["example_id"]: r["answer"] for r in read_jsonl(SFT_DIR / "val_fluency_references_sampled.jsonl")}
model, tokenizer = grpo.load_grpo_model(MODEL, SFT_DIR / "lora_adapter_final", RUN_DIR / "lora_adapter_final")
sft.evaluate(RUN_DIR, val_rows, model, tokenizer, val_refs, method="grpo", scorer=scorer, batch_size=BATCH_SIZE)
del model; gc.collect(); torch.cuda.empty_cache()

display(Markdown(sft.results_table(
    BASELINES_DIR / "eval_answers_no_secret_scored.jsonl", BASELINES_DIR / "eval_answers_prompted_scored.jsonl",
    BASELINES_DIR / "eval_answers_constrained_scored.jsonl",
    SFT_DIR / "val_answers_sft_model_scored.jsonl", DPO_DIR / "val_answers_dpo_model_scored.jsonl", RUN_DIR / "val_answers_grpo_model_scored.jsonl",
)))
print(e0.show_examples(RUN_DIR / "val_answers_grpo_model_scored.jsonl", n=3))

In [ ]:
# Val curve: scores at each snapshot (step-150 is the final model).
snap_paths = grpo.eval_snapshots(RUN_DIR, MODEL, SFT_DIR / "lora_adapter_final", val_rows, scorer, val_refs, batch_size=BATCH_SIZE)
display(Markdown(sft.results_table(SFT_DIR / "val_answers_sft_model_scored.jsonl", *snap_paths)))
free_gpu()

## Reading the results

- **grpo vs dpo:** same starting model, same questions, same answers per question. The difference is online sampling over many rounds against one offline round.
- **Val curve:** if scores still rise from step 100 to 150, more steps would likely help; if they flatten or fall, 150 was enough.
- **fluency** is now part of what's optimised, so watch whether it holds up while the letters improve.